# 07 — Video annotation with GPT-6 Luna

This notebook orchestrates temporal annotation without durably copying the video frames. For each case, the `gpt-6-luna` model receives `previous`, `current` and `next`, then returns either a tight box in `current` or a negative decision.

The workflow is deliberately split into five actions:

1. **plan**: local, free verification;
2. **submit**: temporary upload of the images and launch of the Batch API;
3. **status**: reading the progress;
4. **collect**: download and conversion into YOLO labels;
5. **cleanup**: deleting the remote files after local verification.

> `Run all` triggers no paid call: `SUBMIT_BATCH` and `DELETE_REMOTE_FILES` stay `False` by default.

In [ ]:
import json
import os
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Start with the pilot. Only switch to the full file after inspection.
CONFIG_PATH = PROJECT_ROOT / 'configs/luna_video_pilot.toml'
# CONFIG_PATH = PROJECT_ROOT / 'configs/luna_video_full.toml'

from box2boxcam.data.luna_batch import (
    build_plan,
    cleanup_remote_files,
    collect_batch,
    load_config,
    refresh_status,
    submit_batch,
)

config = load_config(CONFIG_PATH, PROJECT_ROOT)
config

## 1. Build and read the plan

This step uses neither the key nor the API. The plan is immutable: changing the video, the model or the sampling requires another `output` folder.

In [ ]:
plan = build_plan(config)
display(Markdown(
    f"**Video:** `{plan['source_video']}`  \
"
    f"**Cases to annotate:** {plan['case_count']:,}  \
"
    f"**Unique remote images:** {plan['unique_uploaded_frame_count']:,}  \
"
    f"**Model:** `{plan['model']}` — reasoning `{plan['reasoning_effort']}` — detail `{plan['image_detail']}`  \
"
    f"**Local output:** `{config.output_root}`"
))

## 2. Check the local key

The `.env` file must contain `OPENAI_API_KEY=...`. The value is never displayed.

In [ ]:
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env', override=False)
key_configured = bool(os.getenv('OPENAI_API_KEY'))
display(Markdown(
    '✅ API key detected.' if key_configured
    else '❌ Key missing: fill in the `.env` file before submitting.'
))

## 3. Submit the batch

This cell is the only one that sends all the images and starts the paid computation. It is resumable: an interruption keeps every `file_id` already uploaded.

In [ ]:
SUBMIT_BATCH = False

if SUBMIT_BATCH:
    state = submit_batch(config)
    display(Markdown(f"Batch `{state['batch_id']}` : **{state['batch_status']}**"))
else:
    display(Markdown("Submission disabled. Set `SUBMIT_BATCH = True` only when you are ready."))

## 4. Check progress, then collect

`status` does not restart any computation. When the status becomes `completed`, the collection downloads the responses, validates them and writes `predictions.json`, `manifest.csv` and the YOLO labels.

In [ ]:
state_path = config.output_root / 'state.json'
if state_path.exists() and json.loads(state_path.read_text()).get('batch_id'):
    state = refresh_status(config)
else:
    state = None
    display(Markdown('No batch submitted for this run.'))

In [ ]:
if state and state['batch_status'] == 'completed':
    summary = collect_batch(config)
    display(summary)
elif state:
    display(Markdown(f"Batch still **{state['batch_status']}**: come back and run these cells later."))

## 5. Minimal visual inspection

The box is thin and no number hides the ball. The frames are re-read directly from the video: no extra JPEG gallery is stored.

In [ ]:
import cv2
import matplotlib.pyplot as plt

predictions_path = config.output_root / 'predictions.json'
if predictions_path.exists():
    predictions = json.loads(predictions_path.read_text())['predictions']
    shown = predictions[:12]
    capture = cv2.VideoCapture(str(config.video_path))
    fig, axes = plt.subplots(3, 4, figsize=(16, 9))
    for axis, prediction in zip(axes.flat, shown):
        capture.set(cv2.CAP_PROP_POS_FRAMES, prediction['source_frame'])
        ok, frame = capture.read()
        if not ok:
            axis.set_visible(False)
            continue
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axis.imshow(image)
        bbox = prediction['bbox_current_pixels']
        if bbox:
            x1, y1, x2, y2 = bbox
            axis.add_patch(plt.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor='#00ff66', linewidth=1.2,
            ))
        state_label = 'ball' if bbox else 'negative'
        axis.set_title(
            f"frame {prediction['source_frame']} — {state_label} — "
            f"conf {prediction['confidence']:.2f}", fontsize=9
        )
        axis.axis('off')
    capture.release()
    for axis in axes.flat[len(shown):]:
        axis.set_visible(False)
    fig.tight_layout()
    plt.show()
else:
    display(Markdown('No result collected yet.'))

## 6. Clean up the remote files

Only do this after checking that `predictions.json` exists and opens correctly. The local results and the video are not deleted.

In [ ]:
DELETE_REMOTE_FILES = False

if DELETE_REMOTE_FILES:
    deleted = cleanup_remote_files(config)
    display(Markdown(f"{deleted:,} API files deleted."))
else:
    display(Markdown('Remote cleanup disabled.'))

## 7. Prepare the six new videos — offline only

This section is independent of the batch above: it does not load the key and calls neither Luna nor any other model. The candidate frames are extracted before being annotated.

**Stride kept: one frame out of four**, with no cap of 100 frames. Original resolution, JPEG quality 95. Note: at 30 FPS this gives 7.5 frames/second, versus 15 at 60 FPS; it is not a uniform temporal rate.

Each frame keeps its video, its uploader, its URL, its frame number and its presentation timestamp. The identifiers include the video: frames from two clips cannot replace each other.

The match_id, scene_id and split columns are still empty. A video is not automatically an independent match: clips and replays of the same match will have to be grouped before creating the splits. No empty annotation is produced: **unannotated does not mean no ball**.

The cell can be re-run with the same parameters without copying the images again. To change the stride, use a new output folder; no existing work is overwritten.

In [ ]:
import csv
import json
from pathlib import Path

from IPython.display import Markdown, display

from box2boxcam.data.extract_video_frames import extract_video_frames

NEW_PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
NEW_CONFIG = json.loads((NEW_PROJECT_ROOT / "configs/new_video_frames.json").read_text())
FRAME_STRIDE = NEW_CONFIG["frame_stride"]
JPEG_QUALITY = NEW_CONFIG["jpeg_quality"]
NEW_OUTPUT_ROOT = NEW_PROJECT_ROOT / NEW_CONFIG["output_root"]
new_report = extract_video_frames(
    project_root=NEW_PROJECT_ROOT,
    download_manifest=NEW_PROJECT_ROOT / NEW_CONFIG["download_manifest"],
    output_root=NEW_OUTPUT_ROOT,
    frame_stride=FRAME_STRIDE,
    jpeg_quality=JPEG_QUALITY,
)
with (NEW_OUTPUT_ROOT / "manifest.csv").open() as stream:
    new_frames = list(csv.DictReader(stream))
lines = ["| Video | Uploader | FPS | Frames kept |", "|---|---|---:|---:|"]
for video in new_report["videos"]:
    lines.append(f"| {video['video_id']} | {video['uploader']} | {video['fps']:.2f} | {video['images']} |")
display(Markdown("\n".join(lines)))
display(Markdown(
    f"**Total: {new_report['images']} unannotated frames** — "
    f"{new_report['image_bytes']/1024**2:.1f} MiB — no API call.\n\n"
    f"Output: {NEW_OUTPUT_ROOT}"
))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as DisplayImage
from PIL import Image as PILImage

fig, axes = plt.subplots(2, 3, figsize=(9, 11))
for ax, video in zip(axes.flat, new_report["videos"]):
    candidates = [row for row in new_frames if row["video_id"] == video["video_id"]]
    row = candidates[len(candidates)//2]
    with PILImage.open(NEW_PROJECT_ROOT / row["image_path"]) as image:
        image.thumbnail((300, 540))
        pixels = np.asarray(image).copy()
    ax.imshow(pixels)
    ax.set_title(f"{video['uploader']}\n{video['video_id']}\nFrame {row['source_frame']} · {float(row['timestamp_seconds']):.2f} s", fontsize=8)
    ax.axis("off")
fig.suptitle("One frame per new video — no box proposed")
fig.tight_layout()
fig.subplots_adjust(hspace=.3)
preview_path = NEW_OUTPUT_ROOT / "preview.png"
fig.savefig(preview_path, dpi=110)
plt.close(fig)
display(DisplayImage(filename=str(preview_path)))

The 782 images are sent to GPT-6 Luna with **max** reasoning, with three original images (previous/current/next), JPEG 95 and a cap of **32,768 tokens per response, reasoning included**. This cap is not a consumption imposed. Configuration: `configs/luna_new_videos_20260928.json`. Six separate batches prevent temporal neighbors from crossing a video boundary. The process runs outside the notebook, retrieves the responses and does not automatically retry paid failures. The proposals remain to be reviewed: nothing is added to luna_v001 or used for training.

In [ ]:
# Local read only: no API call or paid launch.
import json
from pathlib import Path

from IPython.display import Markdown, display

annotation_root = Path('data/interim/gpt6_luna_new_videos_20260928')
lines = ['| Video | Frames sent | Status | Responses |', '|---|---:|---|---|']
for plan_path in sorted(annotation_root.glob('*/plan.json')):
    plan = json.loads(plan_path.read_text())
    state_path = plan_path.parent / 'state.json'
    state = json.loads(state_path.read_text()) if state_path.exists() else {}
    uploaded = len(state.get('uploaded_images', {}))
    required = plan['unique_uploaded_frame_count']
    status = state.get('batch_status') or 'upload in progress / pending'
    counts = state.get('request_counts', {})
    lines.append(f'| {plan_path.parent.name} | {uploaded}/{required} | {status} | {counts} |')
display(Markdown('\n'.join(lines)))
summary_path = annotation_root / 'summary.json'
if summary_path.exists():
    display(json.loads(summary_path.read_text()))
log_path = annotation_root / 'run.log'
if log_path.exists():
    print('\n'.join(log_path.read_text().splitlines()[-12:]))

## 8. Precision pilot: zoom only or zoom with reference marks
20 positive images spread across the six videos, 40 GPT-6 Luna max responses. The initial proposals are only used to choose a wide area; they are not presented as ground truth. The zoom keeps a margin, is enlarged and receives an identical white border in both variants. The reference marks are thin, the numbers stay within this border, and the coordinate conversion back is done by the code.
Authorized budget: USD 0.50 excluding tax. Before submission, the API counts the inputs; the calculation assumes that each response could use the full 32,768 authorized tokens and that all inputs could be billed as cache writes. The batch is refused if this maximum estimate exceeds the budget. No automatic paid retry.
There were no human corrections submitted at launch time: this is a visual comparison, **not an accuracy measurement**. The Label Studio project and its annotations are never replaced.

In [ ]:
# Local read only. PAGE can be 1 to 5.
import json
from pathlib import Path

from IPython.display import Image, display

PAGE = 1
pilot_root = Path('data/interim/gpt6_luna_crop_pilot20_20260929')
for name in ['state.json', 'summary.json']:
    path = pilot_root / name
    if path.exists():
        obj = json.loads(path.read_text())
        if name == 'state.json':
            display({key: obj.get(key) for key in ['batch_status', 'request_counts', 'budget_check']})
        else:
            display(obj)
comparison = pilot_root / f'comparison_page_{PAGE:02d}.png'
if comparison.exists():
    display(Image(filename=str(comparison)))
else:
    print('Comparisons not available yet. Do not re-run the notebook API cells.')